In [1]:
import polars as pl
import numpy as np
from pathlib import Path

cwd = Path.cwd().parent

excel_path = Path(cwd, "data/Airlink - UMichiagn - Data Collection - 9.8.2025.xlsx")

data = pl.read_excel(excel_path, has_header=False, infer_schema_length=0)

df_clean = (
    data.with_columns(
        pl.col("column_1").str.extract(r"^(\d{4})", 1)
        .forward_fill()
        .alias("Year")
    )
    .filter(
        ~pl.col("column_1").str.contains(r"^\d{4}.*Completed Shipments"), # Remove "2025 Completed..."
        pl.col("column_1") != "NGO ID"                                    # Remove repeated header rows
    )
)

header_row = data.row(1)
rename_map = {
    f"column_{i+1}": name
    for i , name in enumerate(header_row)
    if name and name != "null"
}

df_final = (
    df_clean.rename(rename_map)
)

# display all rows
# pl.Config.set_tbl_rows(-1)
# df_final

In [ ]:
# 1. Identify groups with the same Origin/Destination but multiple Aircraft Types
# 2. Filter for those groups and sort for comparison
comparison_df = (
    df_final
    .filter(
        pl.col("Aircraft Type").n_unique().over(["Origin", "Destination"]) > 1
    )
    .select([
        "Origin", 
        "Destination", 
        "Aircraft Type", 
        "AW (kg)", 
        "Year"
    ])
    .sort(["Origin", "Destination", "AW (kg)"])
)

# Display the results
# show all rows in the comparison dataframe
pl.Config.set_tbl_rows(-1)
display(comparison_df)

Origin,Destination,Aircraft Type,AW (kg),Year
str,str,str,str,str
"""""","""""","""Trucking""","""1979.142""","""2025"""
"""""","""""","""Trucking""","""36.741""","""2025"""
"""""","""""","""""","""627882.5014""","""2025"""
"""""","""""","""""","""6606""","""2025"""
"""""","""""","""""","""711009.1346""","""2025"""
"""EWR""","""MUC""","""Widebody""","""1465.95""","""2025"""
"""EWR""","""MUC""","""Widebody""","""195""","""2025"""
"""EWR""","""MUC""","""Narrowbody""","""2563""","""2025"""
"""EWR""","""MUC""","""Widebody""","""3529""","""2025"""


: 

In [2]:
def load_shipping_data(filepath):
    # 1. Read Raw (as strings) to handle mixed headers
    df_raw = pl.read_excel(filepath, has_header=False, infer_schema_length=0)

    # 2. Extract Year & Clean
    df_clean = (
        df_raw
        .with_columns(
            pl.col("column_1")
            .str.extract(r"^(\d{4})", 1)
            .forward_fill()
            .alias("Year")
        )
        .filter(
            ~pl.col("column_1").str.contains(r"^\d{4}.*Completed Shipments"),
            pl.col("column_1") != "NGO ID",
            pl.col("column_1") != "Mirror",
            pl.col("column_2").is_not_null(),
            pl.col("column_2").str.strip_chars() != ""
        )
    )

    # 3. Rename Columns (Mapping from row 1 of raw file)
    #    Adjust logic if your header row index varies
    header_row = df_raw.row(1)
    rename_map = {f"column_{i+1}": name for i, name in enumerate(header_row) if name}
    
    df_final = (
        df_clean
        .rename(rename_map)
        .with_columns([
            pl.col("Year").cast(pl.Int32),
            pl.lit(1.0).alias("s_label") # These are all Positives
        ])
    )
    return df_final

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
def process_features(df_positives):
    # 1. Define the features you want to use for the model
    features = ["AW (kg)", "CBM", "Origin"]
    
    # 2. Extract and Clean
    # We select the columns and handle the String -> Float conversion fundamentally
    df_cleaned = (
        df_positives
        .select(features + ["s_label"])
        .with_columns([
            # strict=False handles empty strings "" by turning them into nulls
            pl.col("AW (kg)").cast(pl.Float64, strict=False),
            pl.col("CBM").cast(pl.Float64, strict=False)
        ])
        # Replace nulls with 0.0 so the neural network doesn't get NaNs
        .with_columns([
            pl.col("AW (kg)").fill_null(0.0),
            pl.col("CBM").fill_null(0.0)
        ])
    )
    
    # 3. Scikit-Learn Preprocessing
    # This creates the One-Hot encoding for 'Origin' and scales the numbers
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), ["AW (kg)", "CBM"]),
            ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ["Origin"])
        ]
    )
    
    # Transform to the final X matrix
    X_np = preprocessor.fit_transform(df_cleaned.to_pandas())
    s_np = df_cleaned["s_label"].to_numpy().astype(np.float32)
    
    return X_np, s_np

import torch
import torch.nn as nn
import torch.optim as optim
class CarrierAcceptProbability(nn.Module):
    def __init__(self, input_dim):
        super(CarrierAcceptProbability, self).__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        pred = torch.sigmoid(self.linear(x))
        return pred
    
def train_step(X, realizations):
    m = CarrierAcceptProbability(X.shape[1])
    opt = optim.Adam(m.parameters(), lr=0.001)
    criterion = nn.BCELoss()

    for _ in range(1000):
        opt.zero_grad()
        loss = criterion(m(X).squeeze(), realizations)
        loss.backward()
        opt.step()
    return m

KeyboardInterrupt: 

In [1]:
df = load_shipping_data(excel_path)
display(df)

NameError: name 'load_shipping_data' is not defined

In [ ]:


X_numpy, s_numpy = process_features(df)

X_tensor = torch.tensor(X_numpy, dtype=torch.float32)
s_labels = torch.tensor(s_numpy, dtype=torch.float32)

# 3. Train PU Model (Using your function)
print(f"Training on {len(X_tensor)} samples with {X_tensor.shape[1]} features...")
pu_model = train_step(X_tensor, s_labels)

# 4. Estimate 'c' and Calibrate
with torch.no_grad():
    # Average prediction on Labeled Positives (s=1)
    c_est = pu_model(X_tensor[s_labels == 1]).mean().item()
    
    print(f"Estimated Labeling Frequency (c): {c_est:.4f}")
    
    # Example Prediction
    sample_input = X_tensor[0:5]
    raw_prob = pu_model(sample_input).flatten()
    calibrated_prob = torch.clamp(raw_prob / c_est, 0, 1)
    
    print("\nRaw Probabilities (s=1):", raw_prob.numpy())
    print("True Probabilities (y=1):", calibrated_prob.numpy())

In [ ]:
data_folder = cwd / "data"

print(f"Looking in: {data_folder}")
print("--- Files Found ---")

for file in data_folder.glob("*.xlsx"):
    print(file.name)

Looking in: m:\OneDrive - Umich\humanitarian-multimodal\src\data
--- Files Found ---
